## Configuración Google Colab
> **Solo si estás en Colab:** ejecuta la siguiente celda para montar Google Drive.
> Si trabajas en local (VS Code / Jupyter), puedes saltártela.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    # Cambia 'archive' por el nombre de la carpeta que subiste a Drive
    RUTA_DRIVE = '/content/drive/MyDrive/archive'
    os.chdir(RUTA_DRIVE)
    print(f"Directorio: {os.getcwd()}")
    # Instalar dependencias
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'seaborn', 'scipy'])
else:
    import os
    print(f"Entorno local — directorio: {os.getcwd()}")

# EDA Proyecto I — Fase 2: Limpieza y Preparación
**DataTalent Solutions S.L.** | Módulo II: Análisis y Visualización de Datos

En esta fase **unimos los 11 CSV en un único DataFrame maestro** y aplicamos todas las técnicas de limpieza necesarias. Cada decisión está documentada con su justificación.

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
print("Librerías cargadas")

## 2. Carga de los 11 archivos CSV

In [ ]:
postings             = pd.read_csv('postings.csv', low_memory=False)
companies            = pd.read_csv('companies/companies.csv', low_memory=False)
company_industries   = pd.read_csv('companies/company_industries.csv')
company_specialities = pd.read_csv('companies/company_specialities.csv')
employee_counts      = pd.read_csv('companies/employee_counts.csv')
benefits             = pd.read_csv('jobs/benefits.csv')
job_industries       = pd.read_csv('jobs/job_industries.csv')
job_skills           = pd.read_csv('jobs/job_skills.csv')
salaries             = pd.read_csv('jobs/salaries.csv')
industries           = pd.read_csv('mappings/industries.csv')
skills_map           = pd.read_csv('mappings/skills.csv')

print(f"11 CSV cargados | postings: {postings.shape}")

## 3. Unión (JOIN) de los 11 CSV en un DataFrame maestro

Construimos el DataFrame integrado siguiendo el esquema de relaciones detectado en la Fase 1.
Las tablas N:M (skills, industrias, beneficios, especialidades) se agregan como listas separadas por comas para mantener **una fila por oferta de empleo**.

In [ ]:
# ── PASO 1: Salarios (1:1 por job_id) ────────────────────────────
sal = salaries[['job_id','max_salary','min_salary','med_salary',
                'pay_period','currency','compensation_type']].copy()
sal.columns = ['job_id','sal_max','sal_min','sal_med',
               'sal_period','sal_currency','sal_comp_type']

df = postings.merge(sal, on='job_id', how='left')
print(f"[1] + salaries:           {df.shape}")

# ── PASO 2: Empresas (N:1 por company_id) ────────────────────────
comp = companies[['company_id','name','company_size',
                  'state','country','city']].copy()
comp.columns = ['company_id','comp_name','comp_size',
                'comp_state','comp_country','comp_city']
df = df.merge(comp, on='company_id', how='left')
print(f"[2] + companies:          {df.shape}")

# ── PASO 3: Conteo empleados (N:1, registro más reciente) ─────────
emp = (employee_counts
       .sort_values('time_recorded', ascending=False)
       .drop_duplicates('company_id')
       [['company_id','employee_count','follower_count']])
df = df.merge(emp, on='company_id', how='left')
print(f"[3] + employee_counts:    {df.shape}")

# ── PASO 4: Industrias de empresa (N:M → lista) ───────────────────
ci_agg = (company_industries
          .groupby('company_id')['industry']
          .apply(lambda x: ', '.join(x.dropna().unique()))
          .reset_index()
          .rename(columns={'industry': 'comp_industries'}))
df = df.merge(ci_agg, on='company_id', how='left')
print(f"[4] + company_industries: {df.shape}")

# ── PASO 5: Especialidades de empresa (N:M → lista) ───────────────
cs_agg = (company_specialities
          .groupby('company_id')['speciality']
          .apply(lambda x: ', '.join(x.dropna().unique()))
          .reset_index()
          .rename(columns={'speciality': 'comp_specialities'}))
df = df.merge(cs_agg, on='company_id', how='left')
print(f"[5] + company_specialities:{df.shape}")

# ── PASO 6: Industrias de oferta (N:M → lista con nombre) ─────────
ji = job_industries.merge(industries, on='industry_id', how='left')
ji_agg = (ji.groupby('job_id')['industry_name']
            .apply(lambda x: ', '.join(x.dropna().unique()))
            .reset_index()
            .rename(columns={'industry_name': 'job_industries_list'}))
df = df.merge(ji_agg, on='job_id', how='left')
print(f"[6] + job_industries:     {df.shape}")

# ── PASO 7: Skills de oferta (N:M → lista con nombre completo) ────
js = job_skills.merge(skills_map, on='skill_abr', how='left')
js_agg = (js.groupby('job_id')['skill_name']
            .apply(lambda x: ', '.join(x.dropna().unique()))
            .reset_index()
            .rename(columns={'skill_name': 'job_skills_list'}))
df = df.merge(js_agg, on='job_id', how='left')
print(f"[7] + job_skills:         {df.shape}")

# ── PASO 8: Beneficios (N:M → lista) ──────────────────────────────
ben_agg = (benefits.groupby('job_id')['type']
            .apply(lambda x: ', '.join(x.dropna().unique()))
            .reset_index()
            .rename(columns={'type': 'benefits_list'}))
df = df.merge(ben_agg, on='job_id', how='left')
print(f"[8] + benefits:           {df.shape}")

print(f"\nDataFrame maestro: {df.shape[0]:,} filas x {df.shape[1]} columnas")

In [ ]:
print("Columnas del DataFrame maestro integrado:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:>2}. {col}")

**Resultado:** Todos los 11 CSV están unidos en un único DataFrame maestro con una fila por oferta de empleo. Las columnas de tipo lista (skills, industrias, beneficios, especialidades) se han agregado como cadenas separadas por comas para mantener la integridad de la estructura tabular.

## 4. Filtrado: roles relacionados con datos
**Justificación:** El análisis se centra en roles de datos. Filtramos por palabras clave en el título para obtener un subconjunto relevante y reducir el ruido del análisis.

In [ ]:
data_keywords = [
    'data', 'analyst', 'scientist', 'engineer', 'machine learning',
    'analytics', 'intelligence', 'statistician', 'quantitative',
    'etl', 'warehouse', 'big data', 'nlp', 'sql developer',
    'python developer', 'ai engineer', 'ml engineer'
]

mask = df['title'].str.lower().str.contains(
    '|'.join(data_keywords), na=False, regex=True
)
df_data = df[mask].copy()

print(f"Ofertas totales en el dataset:   {len(df):>8,}")
print(f"Roles relacionados con datos:    {len(df_data):>8,}  ({len(df_data)/len(df)*100:.1f}%)")
print(f"\nTop 15 títulos más frecuentes en roles de datos:")
print(df_data['title'].value_counts().head(15).to_string())

## 5. Normalización de textos (.str.lower, .str.strip, .str.replace)
**Justificación:** Sin normalización, "Data Engineer", "data engineer" y " Data Engineer " se tratan como 3 categorías distintas, inflando artificialmente la cardinalidad y distorsionando las frecuencias.

In [ ]:
text_cols = ['title', 'company_name', 'location', 'comp_name',
             'comp_city', 'comp_country',
             'formatted_work_type', 'formatted_experience_level']

for col in text_cols:
    if col in df_data.columns:
        df_data[col] = (df_data[col]
                        .astype(str)
                        .str.lower()           # minúsculas
                        .str.strip()           # eliminar espacios extremos
                        .str.replace(r'\s+', ' ', regex=True)  # espacios múltiples
                        .replace('nan', np.nan))

print("Normalización aplicada: .lower() + .strip() + .replace(espacios)")
print("\nTítulos únicos (top 10) tras normalización:")
print(df_data['title'].value_counts().head(10).to_string())

## 6. Normalización salarial — conversión a salario anual (USD)
El dataset mezcla salarios en distintas frecuencias de pago (hourly, monthly, yearly). Normalizamos todo a **salario anual**.

**Justificación:** Sin normalizar, un salario horario de $40 aparece idéntico a uno anual de $40, distorsionando por completo cualquier análisis comparativo. Las columnas `sal_*` provienen de `salaries.csv` (más completo); si no existen, usamos las de `postings.csv`.

In [ ]:
HORAS_ANIO = 2080  # 40 h/semana × 52 semanas

def salario_anual(row):
    # Prioridad: sal_med > sal_max > sal_min > campos de postings
    for campo in ['sal_med','sal_max','sal_min','med_salary','max_salary','min_salary']:
        val = row.get(campo)
        if not pd.isna(val):
            break
    else:
        return np.nan

    period = str(row.get('sal_period', row.get('pay_period', ''))).upper()
    if 'HOUR' in period:   return val * HORAS_ANIO
    if 'MONTH' in period:  return val * 12
    if 'WEEK'  in period:  return val * 52
    return val  # YEARLY u otro -> se toma como anual

df_data['salary_annual'] = df_data.apply(salario_anual, axis=1)

n_con = df_data['salary_annual'].notna().sum()
print(f"Registros con salario calculado: {n_con:,}  ({n_con/len(df_data)*100:.1f}%)")
print(f"\nEstadísticas del salario anual (USD):")
print(df_data['salary_annual'].describe().round(0).to_string())

## 7. Tratamiento de valores nulos

| Variable | Estrategia | Justificación |
|----------|-----------|---------------|
| `formatted_experience_level` | Imputación con **moda** | Ordinal; el nivel más frecuente es representativo |
| `views`, `applies` | Imputación con **mediana** | Distribución sesgada → mediana es robusta |
| `company_name` / `comp_name` | Relleno con `'Desconocida'` | Evitar pérdida de filas con datos válidos en otras columnas |
| `salary_annual` | **Sin imputación** — análisis solo con registros reales | Inventar salarios distorsionaría las conclusiones y violaría la integridad analítica |
| Columnas con >85% nulos | Excluidas del análisis estadístico principal | No aportan información suficiente para inferencias válidas |

In [ ]:
# 1. Nivel de experiencia → moda
moda_exp_s = df_data['formatted_experience_level'].mode()
moda_exp   = moda_exp_s[0] if len(moda_exp_s) > 0 else 'mid-senior level'
df_data['formatted_experience_level'] = df_data['formatted_experience_level'].fillna(moda_exp)
print(f"experience_level → imputado con moda: '{moda_exp}'")

# 2. Vistas y solicitudes → mediana
for col in ['views', 'applies']:
    med = df_data[col].median()
    df_data[col] = df_data[col].fillna(med)
    print(f"{col} → imputado con mediana: {med:.0f}")

# 3. Nombre de empresa
df_data['company_name'] = df_data['company_name'].fillna('Desconocida')

# 4. Subconjunto con salario (para análisis salarial)
df_sal = df_data[df_data['salary_annual'].notna()].copy()
print(f"\nRegistros con salario disponible: {len(df_sal):,} ({len(df_sal)/len(df_data)*100:.1f}%)")

# Verificar resultado
print("\nNulos restantes en columnas clave:")
cols_check = ['formatted_experience_level','views','applies','company_name','salary_annual']
for c in cols_check:
    if c in df_data.columns:
        n = df_data[c].isna().sum()
        print(f"  {c}: {n}")

## 8. Detección de outliers en salario

Usamos dos métodos complementarios:
- **IQR (Rango Intercuartílico):** robusto, no asume distribución
- **Z-score:** basado en la distancia a la media en desviaciones estándar

In [ ]:
salarios = df_sal['salary_annual'].dropna()

# ── Método 1: IQR ─────────────────────────────────────────────────
Q1  = salarios.quantile(0.25)
Q3  = salarios.quantile(0.75)
IQR = Q3 - Q1
lim_inf = max(Q1 - 1.5 * IQR, 10_000)   # mínimo razonable: $10k/año
lim_sup = Q3 + 1.5 * IQR

out_iqr = df_sal[(df_sal['salary_annual'] < lim_inf) |
                 (df_sal['salary_annual'] > lim_sup)]

print("=== Método IQR ===")
print(f"  Q1  (P25) = ${Q1:>12,.0f}")
print(f"  Q3  (P75) = ${Q3:>12,.0f}")
print(f"  IQR       = ${IQR:>12,.0f}")
print(f"  Límite inferior: ${lim_inf:>10,.0f}")
print(f"  Límite superior: ${lim_sup:>10,.0f}")
print(f"  Outliers IQR: {len(out_iqr):,}  ({len(out_iqr)/len(df_sal)*100:.1f}%)")

In [ ]:
# ── Método 2: Z-score ─────────────────────────────────────────────
z_scores = np.abs(stats.zscore(df_sal['salary_annual']))
out_z = df_sal[z_scores > 3]

print("=== Método Z-score (umbral ±3σ) ===")
print(f"  Media  = ${df_sal['salary_annual'].mean():>10,.0f}")
print(f"  Std    = ${df_sal['salary_annual'].std():>10,.0f}")
print(f"  Outliers (|z| > 3): {len(out_z):,}  ({len(out_z)/len(df_sal)*100:.1f}%)")
print()
print("Salarios más extremos (top 5):")
print(df_sal.nlargest(5,'salary_annual')[['title','comp_name','salary_annual']].to_string(index=False))

**Decisiones documentadas:**

| Tipo de outlier | Criterio | Acción | Justificación |
|----------------|---------|--------|---------------|
| Salario < $10.000/año | Umbral mínimo | **Eliminar** | Probablemente un salario horario mal registrado como anual |
| Salario > límite IQR superior | Método IQR | **Eliminar** | Valores inverosímiles; errores de captura |
| Salarios altos legítimos dentro del IQR | — | **Conservar** | Roles directivos o en FAANG justifican salarios elevados reales |

In [ ]:
# Aplicar filtro de outliers
df_clean = df_sal[
    (df_sal['salary_annual'] >= lim_inf) &
    (df_sal['salary_annual'] <= lim_sup)
].copy()

print(f"Con salario (antes de filtrar outliers): {len(df_sal):>7,}")
print(f"Con salario (después de filtrar):        {len(df_clean):>7,}")
print(f"Outliers eliminados:                     {len(df_sal)-len(df_clean):>7,}")
print(f"\nRango salarial limpio: ${df_clean['salary_annual'].min():,.0f} — ${df_clean['salary_annual'].max():,.0f}")

## 9. Guardado de datasets procesados

In [ ]:
# Dataset maestro completo (todos los roles, todos los joins)
df.to_csv('data_maestro_completo.csv', index=False)
print(f"Guardado: data_maestro_completo.csv     — {df.shape}")

# Dataset filtrado a roles de datos (todos, sin restricción salarial)
df_data.to_csv('data_roles_completo.csv', index=False)
print(f"Guardado: data_roles_completo.csv       — {df_data.shape}")

# Dataset de roles de datos CON salario limpio (para análisis salarial)
df_clean.to_csv('data_roles_salario.csv', index=False)
print(f"Guardado: data_roles_salario.csv        — {df_clean.shape}")

## 10. Resumen de decisiones de limpieza

| Paso | Acción | Justificación |
|------|--------|---------------|
| **Joins** | 11 CSV unidos → 1 DataFrame maestro | Análisis integrado con toda la información disponible |
| **Filtro roles** | Keywords en `title` | Enfocar el análisis en roles de datos |
| **Normalización texto** | `.lower()` + `.strip()` + `.replace()` | Estandarizar categorías para evitar duplicados por capitalización |
| **Salario anual** | Unificación de unidades (hourly→anual, monthly→anual) | Comparar salarios en la misma unidad |
| **experience_level** | Imputación con moda | Categoría ordinal; el nivel más común es representativo |
| **views / applies** | Imputación con mediana | Robustez frente a distribución asimétrica |
| **salary_annual** | Sin imputar; análisis solo con datos reales | Inventar salarios genera conclusiones falsas |
| **Outliers salariales** | Filtro IQR + umbral mínimo $10k | Eliminar errores de entrada; conservar casos legítimos |

**Próximo paso (Fase 3):** Análisis estadístico descriptivo, correlaciones, groupby y detección de sesgos.